#利用VGG/googlenet/resnet 等结构对CIFAR-10进行分类

导入相关包

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import torchvision as tv
import torchvision.transforms as transforms
import torch.nn as nn

定义是否使用GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


数据准备，尝试利用数据增强和归一化，
提示：transforms.RandomCrop(),transforms.RandomHorizontalFlip(),
transforms.Normalize()等函数
可以设计函数获取数据的均值和方差（可选）

In [ ]:
def get_mean_and_std(dataset):
    '''
        可以设计函数获取数据的均值和方差
    '''
    dataloader = torch.utils.data.DataLoader(dataset,batch_size=128,shuffle=False,num_workers=0)
    mean = torch.zeros(3)
    std =torch.zeros(3)
    nb_samples = 0;
    for images,_ in dataloader:
      batch_samples = images.size(0)
      images = images.view(batch_samples,image.size(1),-1)
      mean+= images.mean(2).sum(0)
      std += images.std(2).sum(0)
      nb_samples += batch_samples

    mean /= nb_samples
    std /= nb_samples
    return mean.tolist(),std.tolist()
transform_train =transforms_train = transforms.Compose([
    transforms.RandomCrop(32,padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])
transform_test =transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

### 加载数据，注意文件夹位置

In [ ]:
Batch_size = 128

trainset = tv.datasets.CIFAR10(root='/data/cifar10', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=Batch_size, shuffle=True, num_workers=2)

testset = tv.datasets.CIFAR10(root='/data/cifar10', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=Batch_size, shuffle=False, num_workers=2)

print("数据加载完成！训练集:", len(trainset), "测试集:", len(testset))

100%|██████████| 170M/170M [00:04<00:00, 42.1MB/s]


数据加载完成！训练集: 50000 测试集: 10000


定义网络结构，选取VGG/googlenet/resnet 等网络结构
### 自己定义网络结构，不要直接调用

In [ ]:
class YourNet(nn.Module):
    '''
        设计卷积神经网络
    '''
    def __init__(self):
      super(YourNet,self).__init__()
      self.features = nn.Sequential(
          nn.Conv2d(3,64,kernel_size=3,padding=1),
          nn.BatchNorm2d(64),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2,stride=2),

          nn.Conv2d(64,128,kernel_size=3,padding=1),
          nn.BatchNorm2d(128),
          nn.ReLU(inplace=True),
          nn.Conv2d(128,128,kernel_size=3,padding=1),
          nn.BatchNorm2d(128),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2,stride=2),

          nn.Conv2d(128,256,kernel_size=3,padding=1),
          nn.BatchNorm2d(256),
          nn.ReLU(inplace=True),
          nn.Conv2d(256,256,kernel_size=3,padding=1),
          nn.BatchNorm2d(256),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2,stride=2),



      )

      self.classifier = nn.Sequential(
          nn.Linear(256 * 4 * 4,512),
          nn.ReLU(inplace=True),
          nn.Dropout(0.5),
          nn.Linear(512,512),
          nn.ReLU(inplace=True),
          nn.Dropout(0.5),
          nn.Linear(512,10)
      )
    def forward(self,x):
      x = self.features(x)
      x = x.view(x.size(0),-1)
      x = self.classifier(x)
      return x

实例化网络，尝试加载保存过的模型继续训练，torch.load()函数

In [ ]:
net = YourNet().to(device)
print(net)

YourNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): Conv2d(2

In [ ]:
start_epoch = 0
best_acc = 0.0
checkpoint_path = 'best_model.pth'

try:
  checkpoint = torch.load(checkpoint_path,map_location=device)
  net.load_state_dict(checkpoint['net'])
  best_acc = checkpoint['acc']
  start_epoch = checkpoint['epoch'] + 1
  print("成功加载之前保存的模型！")
  print(f"当前最佳准确率: {best_acc:.2f}%")
  print(f"将从 epoch {start_epoch} 开始继续训练")

except FileNotFoundError:
  print("未找到已保存模型 (best_model.pth)，将从头开始训练...")
except Exception as e:
  print(f"加载模型时出现错误: {e}")
  print("将从头开始训练...")

未找到已保存模型 (best_model.pth)，将从头开始训练...


超参数设置，定义损失函数和优化方式

In [ ]:
EPOCH = 200
Batch_size = 128
LR = 0.001
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    net.parameters(),
    lr = LR,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max = EPOCH)

print("超参数设置完成！")
print(f"总训练轮数 EPOCH = {EPOCH}")
print(f"Batch Size = {Batch_size}")
print(f"初始学习率 LR = {LR}")
print(f"优化器 = SGD + Momentum + Weight Decay")
print(f"损失函数 = CrossEntropyLoss")

超参数设置完成！
总训练轮数 EPOCH = 200
Batch Size = 128
初始学习率 LR = 0.001
优化器 = SGD + Momentum + Weight Decay
损失函数 = CrossEntropyLoss


## 训练

In [ ]:
def train(epoch):
    '''
        Your code
    '''
    net.train()
    train_loss = 0.0
    correct =0
    total =0

    for batch_idx,(inputs,targets) in enumerate(trainloader):
      inputs,targets= inputs.to(device),targets.to(device)
      optimizer.zero_grad()
      outputs = net(inputs)
      loss = criterion(outputs,targets)
      loss.backward()
      optimizer.step()

      train_loss += loss.item()
      _,predicted = outputs.max(1)
      total += targets.size(0)
      correct += predicted.eq(targets).sum().item()

      if batch_idx % 100 ==0:
        print(f"Train Epoch: {epoch} [{batch_idx * len(inputs)}/{len(trainloader.dataset)}]"
           f"Loss: {loss.item():.4f}")

    avg_loss = train_loss / len(trainloader)
    acc = 100.* correct/total

    print(f"【Train】 Epoch: {epoch}  |  Loss: {avg_loss:.4f}  |  Acc: {acc:.2f}%")


## 测试
    

In [ ]:
def test(epoch):
    '''
        注意：保存当前训练最优模型
    '''
    global best_acc
    net.eval()
    test_loss =0.0
    correct = 0
    total = 0

    with torch.no_grad():
      for batch_idx, (inputs,targets) in enumerate(testloader):
        inputs,targets = inputs.to(device),targets.to(device)

        outputs = net(inputs)
        loss = criterion(outputs,targets)

        test_loss += loss.item()
        _,predicted = outputs.max(1)
        total += targets.size(0)
        correct+= predicted.eq(targets).sum().item()

    avg_loss = test_loss / len(testloader)
    acc = 100.* correct / total

    print(f"【测试】 Epoch: {epoch}  |  Loss: {avg_loss:.4f}  |  Acc: {acc:.2f}%")

    global best_acc
    if acc> best_acc:
      print(f'保存新最优模型！准确率从 {best_acc:.2f}% → {acc:.2f}%')
      best_acc = acc

      state ={
          'net':net.state_dict(),
          'acc':acc,
          'epoch':epoch,
      }
      torch.save(state,'best_model.pth')
      print(f"模型已保存为: best_model.pth\n")
    else:
      print(f"当前准确率 {acc:.2f}%，未超过最佳值 {best_acc:.2f}%\n")


In [ ]:
# 训练、测试
best_acc = 0.0
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()
print("="*60)
print(f"训练全部完成！最终最佳测试准确率: {best_acc:.2f}%")
print(f"最优模型已保存为: best_model.pth")
print("="*60)


Train Epoch: 0 [0/50000]Loss: 2.3283
Train Epoch: 0 [12800/50000]Loss: 2.1168
Train Epoch: 0 [25600/50000]Loss: 1.8046
Train Epoch: 0 [38400/50000]Loss: 1.8087
【Train】 Epoch: 0  |  Loss: 1.9364  |  Acc: 26.09%
【测试】 Epoch: 0  |  Loss: 1.5449  |  Acc: 42.52%
保存新最优模型！准确率从 0.00% → 42.52%
模型已保存为: best_model.pth

Train Epoch: 1 [0/50000]Loss: 1.7353
Train Epoch: 1 [12800/50000]Loss: 1.6081
Train Epoch: 1 [25600/50000]Loss: 1.5300
Train Epoch: 1 [38400/50000]Loss: 1.3562
【Train】 Epoch: 1  |  Loss: 1.5243  |  Acc: 42.26%
【测试】 Epoch: 1  |  Loss: 1.3204  |  Acc: 50.61%
保存新最优模型！准确率从 42.52% → 50.61%
模型已保存为: best_model.pth

Train Epoch: 2 [0/50000]Loss: 1.4039
Train Epoch: 2 [12800/50000]Loss: 1.3276
Train Epoch: 2 [25600/50000]Loss: 1.3181
Train Epoch: 2 [38400/50000]Loss: 1.0449
【Train】 Epoch: 2  |  Loss: 1.3266  |  Acc: 51.08%
【测试】 Epoch: 2  |  Loss: 1.2361  |  Acc: 55.69%
保存新最优模型！准确率从 50.61% → 55.69%
模型已保存为: best_model.pth

Train Epoch: 3 [0/50000]Loss: 1.2403
Train Epoch: 3 [12800/50000]Loss: 